# File 2 — YC Dataset AI Summary

Adds two AI fields per company to the latest **Base** dataset:
`ai_description` (6-7 sentences) and `ai_risks` (1-2 short). Only companies whose
`(id, model, prompt_version)` key is new are summarized — re-runs are cheap.

**Provider switch**: `claude` (default), `groq`, or `mock` (offline, no spend).
**Output switch**: `download` / `drive` / `commit`.

In [ ]:
# --- parameters (papermill overrides these) ---
provider = "claude"       # "claude" | "groq" | "mock"
model = None               # None -> provider default (config)
output = "download"        # "download" | "drive" | "commit"
out_dir = "data"
date = None                 # None -> today
base_path = None            # None -> newest yc_dataset_base_*.parquet in out_dir
cache_path = None           # None -> data/cache/ai_cache.json
drive_folder = "Project YC Scouter"

In [ ]:
# --- ensure the package is importable ---
try:
    import yc_scouter  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=False)
    import yc_scouter  # noqa: F401

In [ ]:
# --- add AI description + risks (only for new cache keys), then dated export ---
from pathlib import Path
from yc_scouter import ai, pipeline

cache = Path(cache_path) if cache_path else ai.DEFAULT_CACHE_PATH
df, paths = pipeline.build_ai(
    base_path=base_path,
    provider=provider,
    model=model,
    cache_path=cache,
    out_dir=Path(out_dir),
    date=date,
)
n_ai = (df["ai_model"] != "").sum()
print(f"AI summary: {len(df)} companies, {n_ai} with AI fields (provider={provider})")
print("Wrote:", paths["parquet"].name, "and", paths["xlsx"].name)

In [ ]:
# --- deliver per the output switch ---
if output == "download":
    try:
        from google.colab import files
        for p in paths.values():
            files.download(str(p))
    except Exception as e:
        print("download skipped (not in Colab):", e)
elif output == "drive":
    try:
        import shutil
        from google.colab import drive
        drive.mount("/content/drive")
        dest = Path(f"/content/drive/MyDrive/{drive_folder}")
        dest.mkdir(parents=True, exist_ok=True)
        for p in paths.values():
            shutil.copy2(p, dest / p.name)
        print("Saved to Drive:", dest)
    except Exception as e:
        print("drive save skipped:", e)
else:  # commit
    print("Files ready in", out_dir, "(GitHub Actions will commit them).")